# Atividade Prática 4


Aluno: João Gabriel Angelo Bradachi

Professor: Fabrício Silva

Objetivo: Aplicar técnicas engenharia de atributos no texto da mensagem para gerar atributos relevantes


In [18]:
%pip install nltk optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 2.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 1.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 2.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 621.4/621.4 kB 2.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.0/80.0 kB 2.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# imports
import pandas as pd

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

import optuna
import xgboost as xgb
from sklearn.model_selection import cross_val_score

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

/home/bradachi/Documentos/gitpath/ML-engineering-final-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Obtendo dados
df_spam = pd.read_csv("spam-dataset.csv")
df_spam

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


## Tratamento dos dados

Para tratar dados textuais, precisamos lidar com a forma que o computador entenda

Vamos realizar 2 tratamentos iniciais: Tokenização e remoção de stop words 

Lematização e Radicalização -> pode ser que piora os dados, se tivermos tempo iremos comparar ao final

## Predição dos dados

Para fazer a predição se é spam ou não, podemos utilizar as seguintes metodologias:

- executar TF-IDF e treinar com o  XGBoost

- Podemos calcular a similaridade de cossenos e ver se o que chegou é mais similar aos que são spam do que com os que não são spam

- usar o Word2Vector para identificar padrões que são parecidos com os spams

In [ ]:
# Tratamento: (Sem Lematização e Radicalização)

stop_words = set(stopwords.words('english'))

# Filtrar as palavras que não estão na lista de stopwords

def remove_sw(texto):
    tokens = word_tokenize(texto.lower())
    filtered_tokens = [word for word in tokens if word not in stop_words]
    clean_text = " ".join(filtered_tokens)
    return clean_text

df_spam['sem_sw'] = df_spam['Message'].apply(remove_sw)
df_spam

,Category,Message,sem_sw
0,ham,"Go until jurong point, crazy.. Available only ...","go jurong point , crazy .. available bugis n g..."
1,ham,Ok lar... Joking wif u oni...,ok lar ... joking wif u oni ...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entry 2 wkly comp win fa cup final tkts 2...
3,ham,U dun say so early hor... U c already then say...,u dun say early hor ... u c already say ...
4,ham,"Nah I don't think he goes to usf, he lives aro...","nah n't think goes usf , lives around though"
...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,2nd time tried 2 contact u. u £750 pound prize...
5568,ham,Will ü b going to esplanade fr home?,ü b going esplanade fr home ?
5569,ham,"Pity, * was in mood for that. So...any other s...","pity , * mood . ... suggestions ?"
5570,ham,The guy did some bitching but I acted like i'd...,guy bitching acted like 'd interested buying s...


In [8]:
# Aplicando TF-IDF

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df_spam['sem_sw'])

nomes_recursos = vectorizer.get_feature_names_out()

df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=nomes_recursos)

df_tfidf

,00,000,000pes,008704050406,0089,0121,01223585236,01223585334,0125698789,02,...,zhong,zindgi,zoe,zogtorius,zoom,zouk,zyada,èn,ú1,〨ud
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5567,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5568,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5569,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5570,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Testando no XGB 

Aqui nesse ponto nós poderiamos pensar em diminuir as dimensões para deixar o treinamento mais rapido (e talvez melhor)

Vamos treinar com tudo, fazer um fine tunning dos hiper-parametros com cross validation

In [ ]:
# Adicionando o label
df_tfidf['Category'] = df_spam['Category']
df_tfidf

,00,000,000pes,008704050406,0089,0121,01223585236,01223585334,0125698789,02,...,zindgi,zoe,zogtorius,zoom,zouk,zyada,èn,ú1,〨ud,Category
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,ham
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,ham
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,spam
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,ham
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,ham
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5567,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,spam
5568,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,ham
5569,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,ham
5570,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,ham


In [21]:
def trata_categoria(categoria):
    if categoria == 'ham': return 0
    elif categoria == 'spam': return 1
    else: return -1

df_tfidf['Category'] = df_tfidf['Category'].apply(trata_categoria)
df_tfidf

,00,000,000pes,008704050406,0089,0121,01223585236,01223585334,0125698789,02,...,zindgi,zoe,zogtorius,zoom,zouk,zyada,èn,ú1,〨ud,Category
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5567,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
5568,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
5569,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
5570,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [ ]:
# Treinamento com cross validation

X = df_tfidf.drop('Category', axis=1) 
y = df_tfidf['Category']              

# Definindo a função objetivo para o Optuna
def objective(trial):
  params = {
      'n_estimators': trial.suggest_int('n_estimators', 50, 300),
      'max_depth': trial.suggest_int('max_depth', 3, 9),
      'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
      'subsample': trial.suggest_float('subsample', 0.5, 1.0),
      'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
      'random_state': 42,
  }

  model = xgb.XGBClassifier(**params)

  # Validação cruzada com 5 folds
  scores = cross_val_score(model, X, y, cv=5, scoring='accuracy', verbose=1)
  return scores.mean()


# Executando o estudo
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

print('Melhores parâmetros:', study.best_params)

[I 2026-09-22 14:12:54,060] A new study created in memory with name: no-name-3074b307-8f6f-432d-b26e-d2e467ea1bd5
[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:  7.9min finished
[I 2026-09-22 14:14:45,186] Trial 3 finished with value: 0.9743360893962691 and parameters: {'n_estimators': 180, 'max_depth': 6, 'learning_rate': 0.062198087303159136, 'subsample': 0.9895914421306642, 'colsample_bytree': 0.9972166529015971}. Best is trial 3 with value: 0.9743360893962691.
[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:  9.5min finished
[I 2026-09-22 14:16:20,753] Trial 4 finished with value: 0.9599781017784255 and parameters: {'n_estimators': 246, 'max_depth': 5, 'learning_rate': 0.011945801925018722, 'subsample': 0.8261087222002826, 'colsample_bytree': 0.5043480485976634}. Best is trial 3 with value: 0.9743360893962691.
[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed: 10.7min finished
[I 2026-09-22 14:17:28,812] Trial 0 finished with value: 0.9775660770785196 and parameters: {'

Melhores parâmetros: {'n_estimators': 295, 'max_depth': 3, 'learning_rate': 0.1880631789682316, 'subsample': 0.9100883248880922, 'colsample_bytree': 0.676551473173548}


## Resultados


In [ ]:
melhores_parametros = {
    'n_estimators': 295,
    'max_depth': 3,
    'learning_rate': 0.1880631789682316,
    'subsample': 0.9100883248880922,
    'colsample_bytree': 0.676551473173548
}

X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, random_state=42)

modelo_pipeline = Pipeline(steps=[
    ('classificador', XGBClassifier(
        n_estimators=295,     # Número de árvores
        max_depth=3,          # Profundidade máxima de cada árvore
        learning_rate=0.1880631789682316,
        subsample=0.9100883248880922,
        colsample_bytree=0.676551473173548,
        random_state=42
    ))
])

print("Treinando o modelo...")
modelo_pipeline.fit(X_treino, y_treino)

print("Fazendo previsões no conjunto de teste...")
previsoes = modelo_pipeline.predict(X_teste)

print("\nRelatório de Classificação:")
print(classification_report(y_teste, previsoes, digits=5))

Treinando o modelo...
Fazendo previsões no conjunto de teste...

Relatório de Classificação:
              precision    recall  f1-score   support

           0    0.98067   0.99793   0.98923       966
           1    0.98485   0.87248   0.92527       149

    accuracy                        0.98117      1115
   macro avg    0.98276   0.93521   0.95725      1115
weighted avg    0.98123   0.98117   0.98068      1115



Devido o relatório apresentar bons resultados nas métricas, não irei testar as outras possibilidades.

## Conclusão

Nesse trabalho foi possível observar e colocar em prática como lidar com dados textuais. A prática foi feita com sucesso, gerando um modelo aceitável com poucas ferramentas. Posteriormente testarei as opções ditas e não testadas nesse trabalho a fim de reduzir o custo computacional (reduzir as dimensões dos dados) e obter resultados semelhantes (ou até melhores) dos obtidos 